In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, TimestampType, FloatType, DateType

catalog_name = 'ecommerce'

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show(10)

In [0]:
df_silver = df_bronze.withColumn('brand_name', F.trim(F.col('brand_name')))
df_silver.show(10)

In [0]:
df_silver = df_silver.withColumn('brand_code', F.regexp_replace(F.col('brand_code'), r'[^A-Za-z0-9]', '' ))
df_silver.show(10)

In [0]:
df_silver.select("category_code").distinct().show()

In [0]:
# anomalies
anomalies = {
    'GROCERY': "GRCY",
    'BOOKS': "BKS",
    'TOYS': "TOY"
}

df_silver = df_silver.replace(anomalies, subset='category_code')
df_silver.select("category_code").distinct().show()



In [0]:
# Write raw data to the silver layer
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands")


### Category

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")
df_bronze.show()

In [0]:
df_duplicates = df_bronze.groupBy("category_code").count().filter(F.col("count") > 1)
display(df_duplicates)

In [0]:
df_silver = df_bronze.dropDuplicates(['category_code'])
display(df_silver)

In [0]:
df_silver = df_silver.withColumn('category_code', F.upper(F.col("category_code")))
display(df_silver)

In [0]:

df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_category")

### Products

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_products")

row_count, col_count = df_bronze.count(), len(df_bronze.columns)

print(f"Found {row_count:,} rows and {col_count} columns in the bronze layer.")


In [0]:
display(df_bronze.limit(5))

In [0]:
df_bronze.select("weight_grams").show(5, truncate=False)

In [0]:
#replace 'g' with '' and convert to float
df_silver = df_bronze.withColumn(
            "weight_grams", 
            F.regexp_replace(F.col("weight_grams"), "g", "").cast(IntegerType())
)
df_silver.select("weight_grams").show(5, truncate=False)

In [0]:
# check length_cm replace comma with dot and convert to float
df_silver.select("length_cm").show(5, truncate=False)

In [0]:
df_silver = df_silver.withColumn(
    "length_cm",
    F.regexp_replace(F.col("length_cm"), ",", ".").cast(FloatType())
)
df_silver.select("length_cm").show(5, truncate=False)


category_code and brand_code are in lower case. we need to make it upper case

In [0]:
df_silver.select("category_code", "brand_code").show(2)

In [0]:
df_silver = df_silver.withColumn(
    "category_code",
    F.upper(F.col("category_code"))
).withColumn(
    "brand_code",
    F.upper(F.col("brand_code"))
)

In [0]:
df_silver.select("category_code", "brand_code").show(2)

In [0]:
df_silver.select("material").distinct().show()

In [0]:
# Fix spelling mistakes
df_silver = df_silver.withColumn(
    "material",
    F.when(F.col("material") == "Coton", "Cotton")
    .when(F.col("material") == "Alumium", "Aluminum")
    .when(F.col("material") == "Ruber", "Rubber")
    .otherwise(F.col("material"))
)

df_silver.select("material").distinct().show()

In [0]:
df_silver.filter(F.col('rating_count')<0).select("rating_count").show(3)

In [0]:
#convert negative rating to positive

df_silver = df_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
    .otherwise(F.lit(0))
)


In [0]:
df_silver.select(
    "weight_grams",
    "length_cm",
    "category_code",
    "brand_code",
    "material",
    "rating_count"
).show(10, truncate=False)

In [0]:
df_silver.write.format("delta") \
        .mode("overwrite") \
        .option("mergeShema", "true") \
        .saveAsTable(f"{catalog_name}.silver.slv_products")

### Customers

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_customers")

row_count, col_count = df_bronze.count(), len(df_bronze.columns)
print(f"Found {row_count:,} rows and {col_count} columns in bronze_customers")

df_bronze.show(5)



Handle null values in customer_id

In [0]:
null_count = df_bronze.filter(F.col("customer_id").isNull()).count()
null_count

In [0]:
df_bronze.filter(F.col("customer_id").isNull()).show(3)

In [0]:
df_silver = df_bronze.dropna(subset=['customer_id'])

row_count = df_silver.count()
print(f"Row count after droping null values: {row_count:,}")

Handle Null Values in phone col

In [0]:
null_count = df_silver.filter(F.col("phone").isNull()).count()
print(f"Null count for phone: {null_count:,}")


In [0]:
df_silver.filter(F.col("phone").isNull()).show(3)

In [0]:
df_silver = df_silver.fillna("Not Available",subset=['phone'])

df_silver.filter(F.col("phone").isNull()).show()

In [0]:

df_silver.write.format("delta") \
        .mode("overwrite") \
        .option("mergeShema", "true") \
        .saveAsTable(f"{catalog_name}.silver.slv_customers")

### Calendar/Date

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_calendar")

row_count, col_count = df_bronze.count(), len(df_bronze.columns)
print(f"Found {row_count:,} rows and {col_count} columns in the bronze table.")

df_bronze.show(3)


In [0]:
df_bronze.printSchema()

Converting String to date

In [0]:
from pyspark.sql.functions import to_date

df_silver = df_bronze.withColumn("date", to_date(df_bronze['date'], "dd-MM-yyyy"))

In [0]:
df_silver.printSchema()

In [0]:
df_silver.show(3)

Remove duplicates

In [0]:
duplicates_count = df_silver.groupBy('date').count().filter("count > 1")
print(f"Found {duplicates_count.count()} duplicate dates in the silver table.")

duplicates_count.show()

In [0]:
df_silver = df_silver.dropDuplicates(['date'])

row_count = df_silver.count()
print("Rows after removing duplicates: ", row_count)

day_name normalize casing

In [0]:
# Capitalize each word in the day_name column
df_silver = df_silver.withColumn("day_name", F.initcap(F.col("day_name")))

df_silver.show(5)


Convert negative week_of_year to positive

In [0]:
df_silver = df_silver.withColumn("week_of_year", F.abs(F.col("week_of_year")))

df_silver.show(5)

Enhance quarter and week_of_year column

In [0]:
df_silver = df_silver.withColumn("quarter", F.concat_ws("",F.concat(F.lit("Q"),F.col("quarter"),F.lit("-"),F.col("year"))))

df_silver = df_silver.withColumn("week_of_year", F.concat_ws("-",F.concat(F.lit("Week"),F.col("week_of_year"),F.lit("-"),F.col("year"))))
df_silver.show(5)

Rename the columns

In [0]:
df_silver = df_silver.withColumnRenamed("week_of_year", "week")


In [0]:
df_silver.write.format("delta") \
        .mode("overwrite") \
        .option("mergeShema", "true") \
        .saveAsTable(f"{catalog_name}.silver.slv_calendar")